In [16]:
import requests
import pandas as pd
import time
from tqdm import tqdm
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ──────────────────────────────────────────────
# НАСТРОЙКИ
# ──────────────────────────────────────────────
API_KEY   = '34185853-958a-419a-8a4b-4079810bae52'   # https://dev.2gis.ru/
LOCATIONS = [
    "76.9286,43.2389",   # Центр
    "76.8500,43.2570",   # Ауэзовский
    "76.9500,43.2900",   # Медеуский
    "77.0200,43.2500",   # Турксибский
    "76.8500,43.1900",   # Наурызбайский
    "76.8700,43.3200",   # Алатауский
]

QUERIES = [
    'парковка',
    'паркинг',
    'автостоянка',
    'Almaty Parking'
]

RADIUS = 15000
PAGE_SIZE = 10
MAX_PAGES = 5
DELAY = 0.5
OUTPUT = 'parking_almaty.xlsx'

FIELDS = (
    'items.point,'
    'items.address_name,'
    'items.name,'
    'items.full_name,'
    'items.schedule,'
    'items.attribute_groups,'
    'items.rubrics,'
    'items.org'
)

BASE_URL = 'https://catalog.api.2gis.com/3.0/items'


# ──────────────────────────────────────────────
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ──────────────────────────────────────────────

def parse_schedule(schedule: dict) -> str:
    if not schedule:
        return ''
    if schedule.get('is_24x7'):
        return 'Круглосуточно'
    days_map = {'Mon':'Пн','Tue':'Вт','Wed':'Ср','Thu':'Чт',
                'Fri':'Пт','Sat':'Сб','Sun':'Вс'}
    parts = []
    for day, info in schedule.get('working_hours', {}).items():
        label = days_map.get(day, day)
        intervals = info.get('working_time', [])
        times = ', '.join(f"{t['from']}-{t['to']}" for t in intervals)
        if times:
            parts.append(f'{label}: {times}')
        elif info.get('is_working_now') is False:
            parts.append(f'{label}: выходной')
    return ' | '.join(parts)


def extract_attributes(attribute_groups: list) -> dict:
    result = {'платная': '', 'тариф': '', 'мест': ''}
    for group in (attribute_groups or []):
        for attr in group.get('attributes', []):
            name = attr.get('name', '').lower()
            val  = attr.get('value', '')
            if isinstance(val, list):
                val = ', '.join(str(v.get('text', v)) if isinstance(v, dict) else str(v) for v in val)
            else:
                val = str(val)
            if any(k in name for k in ['платн', 'бесплатн', 'оплат']):
                result['платная'] = val
            elif any(k in name for k in ['тариф', 'стоимость', 'цена', 'час']):
                result['тариф'] = val
            elif any(k in name for k in ['мест', 'количество', 'вместимость']):
                result['мест'] = val
    return result


def classify_type(item: dict) -> str:
    text = (item.get('name', '') + ' ' + item.get('full_name', '') + ' ' +
            ' '.join(r.get('name', '') for r in item.get('rubrics', []))).lower()
    if any(k in text for k in ['торгов', 'тц', 'молл', 'mall', 'мегацентр']):
        return 'ТЦ'
    if any(k in text for k in ['бизнес-центр', 'бц', 'офисн', 'бизнес центр']):
        return 'БЦ'
    if any(k in text for k in ['городская', 'муниципальн', 'акимат']):
        return 'Городская'
    if any(k in text for k in ['подземн', 'многоуровн', 'паркинг']):
        return 'Паркинг'
    if any(k in text for k in ['жилой', 'жк ', 'residential']):
        return 'ЖК'
    return 'Частная'


def get_paid_status(item: dict, attrs: dict) -> str:
    if attrs['платная']:
        return attrs['платная']
    text = (item.get('name', '') + ' ' +
            ' '.join(r.get('name', '') for r in item.get('rubrics', []))).lower()
    if 'бесплатн' in text:
        return 'Бесплатная'
    if 'платн' in text:
        return 'Платная'
    return ''


def make_link(item: dict) -> str:
    obj_id = item.get('id', '')
    if obj_id:
        return f'https://2gis.kz/almaty/firm/{obj_id}'
    pt = item.get('point', {})
    lat, lon = pt.get('lat'), pt.get('lon')
    if lat and lon:
        return f'https://2gis.kz/almaty?m={lon},{lat}'
    return ''


def get_related_object(item: dict) -> str:
    org = item.get('org', {})
    if org:
        return org.get('name', '')
    full = item.get('full_name', '')
    if ',' in full:
        return full.split(',', 1)[-1].strip()
    return ''


# ──────────────────────────────────────────────
# ПАРСИНГ
# ──────────────────────────────────────────────



        
def fetch_parkings():
    all_items = []

    for location in LOCATIONS:
        print(f"\nЛокация: {location}")

        for query in QUERIES:
            print("Поиск:", query)

            for page in range(1, MAX_PAGES + 1):

                params = {
                    'q': query,
                    'location': location,
                    'radius': RADIUS,
                    'fields': FIELDS,
                    'page': page,
                    'page_size': PAGE_SIZE,
                    'key': API_KEY,
                    'locale': 'ru_KZ',
                }

                success = False

                for attempt in range(3):
                    try:
                        r = requests.get(
                            BASE_URL,
                            params=params,
                            timeout=30
                        )

                        r.raise_for_status()

                        data = r.json()

                        if data.get("meta", {}).get("code") != 200:
                            break

                        items = data.get("result", {}).get("items", [])

                        if not items:
                            break

                        all_items.extend(items)

                        success = True
                        break

                    except requests.exceptions.Timeout:
                        print(
                            f"Timeout. Попытка {attempt+1}/3"
                        )
                        time.sleep(2)

                    except Exception as e:
                        print(e)
                        break

                if not success:
                    continue

                time.sleep(DELAY)

    print("Собрано объектов:", len(all_items))

    return all_items


def build_dataframe(raw: list) -> pd.DataFrame:
    rows = []

    for item in raw:
        pt = item.get('point', {})

        lat = pt.get('lat', '')
        lon = pt.get('lon', '')

        attrs = extract_attributes(
            item.get('attribute_groups', [])
        )

        # ID для удаления дублей
        parking_id = item.get('id', '')

        # если ID нет, используем координаты
        if not parking_id:
            parking_id = f"{lat}_{lon}"

        # название
        name = (
          item.get('name')
          or f"Parking_{item.get('subtype', 'unknown')}_{parking_id}"
        )

        # адрес
        address = item.get('address_name', '')

        if not address:
           address = 'Адрес не указан'

        # координаты
        coordinates = (
          f'{lat}, {lon}'
          if lat and lon
          else ''
        )

        # платная / бесплатная
        paid_status = get_paid_status(item, attrs)
        if not paid_status:
            paid_status = 'Неизвестно'

        # тариф
        tariff = attrs.get('тариф', '')
        if not tariff:
            tariff = 'Неизвестно'

        # количество мест
        places = attrs.get('мест', '')
        if not places:
            places = 'Неизвестно'

        # объект
        related_object = get_related_object(item)
        if not related_object:
            related_object = 'Неизвестно'

        # часы работы
        schedule = parse_schedule(
            item.get('schedule', {})
        )

        if not schedule:
            schedule = 'Неизвестно'

        rows.append({
            'ID': parking_id,
            'Название': name,
            'Адрес': address,
            'Координаты': coordinates,
            'Ссылка 2ГИС': make_link(item),
            'Платная/Бесплатная': paid_status,
            'Тариф': tariff,
            'Кол-во мест': places,
            'Тип парковки': classify_type(item),
            'Объект': related_object,
            'Часы работы': schedule,
        })

    df = pd.DataFrame(rows)

    if df.empty:
        print("Данные не получены.")
        return df

    # удаляем дубликаты по ID
    before = len(df)

    df = df.drop_duplicates(
        subset=['ID'],
        keep='first'
    )

    after = len(df)

    print(
        f'Удалено дубликатов: {before - after}'
    )

    # сортировка
    df = df.sort_values(
        ['Тип парковки', 'Название']
    ).reset_index(drop=True)

    return df



  


def save_excel(df: pd.DataFrame, path: str):
    wb = Workbook()
    ws = wb.active
    ws.title = 'Парковки Алматы'

    thin        = Side(style='thin', color='CCCCCC')
    border      = Border(left=thin, right=thin, top=thin, bottom=thin)
    center      = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left        = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    hdr_fill    = PatternFill('solid', start_color='1F4E79')
    hdr_font    = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    even_fill   = PatternFill('solid', start_color='EBF3FB')
    odd_fill    = PatternFill('solid', start_color='FFFFFF')
    data_font   = Font(name='Arial', size=9)

    # Заголовки
    for col, h in enumerate(df.columns, 1):
        c = ws.cell(row=1, column=col, value=h)
        c.fill = hdr_fill; c.font = hdr_font
        c.alignment = center; c.border = border

    # Данные
    for r_idx, row in enumerate(df.itertuples(index=False), 2):
        fill = even_fill if r_idx % 2 == 0 else odd_fill
        for c_idx, val in enumerate(row, 1):
            c = ws.cell(row=r_idx, column=c_idx, value=str(val) if pd.notna(val) else '')
            c.fill = fill; c.font = data_font
            c.alignment = left; c.border = border

    # Ширина столбцов
    for i, w in enumerate([30, 35, 22, 40, 18, 20, 12, 15, 30, 35], 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    wb.save(path)
    print(f'Сохранено: {path}  ({len(df)} строк)')


# ──────────────────────────────────────────────
# ЗАПУСК
# ──────────────────────────────────────────────

if __name__ == '__main__':
    raw = fetch_parkings()
    df  = build_dataframe(raw)


if df.empty:
    print("Нет данных для сохранения.")
else:
    print(df['Тип парковки'].value_counts())

    print(
        df['Платная/Бесплатная']
        .value_counts()
    )

    save_excel(df, OUTPUT)

    df.to_csv(
        OUTPUT.replace('.xlsx', '.csv'),
        index=False,
        encoding='utf-8-sig'
    )

    print("CSV также сохранён.")


Локация: 76.9286,43.2389
Поиск: парковка
Поиск: паркинг
Поиск: автостоянка
Поиск: Almaty Parking

Локация: 76.8500,43.2570
Поиск: парковка
Поиск: паркинг
Поиск: автостоянка
Поиск: Almaty Parking

Локация: 76.9500,43.2900
Поиск: парковка
Поиск: паркинг
Поиск: автостоянка
Поиск: Almaty Parking

Локация: 77.0200,43.2500
Поиск: парковка
Поиск: паркинг
Поиск: автостоянка
Поиск: Almaty Parking

Локация: 76.8500,43.1900
Поиск: парковка
Поиск: паркинг
Поиск: автостоянка
Поиск: Almaty Parking

Локация: 76.8700,43.3200
Поиск: парковка
Поиск: паркинг
Поиск: автостоянка
Поиск: Almaty Parking
Собрано объектов: 1129
Удалено дубликатов: 529
Тип парковки
Паркинг    335
Частная    247
ТЦ          15
БЦ           3
Name: count, dtype: int64
Платная/Бесплатная
Неизвестно    577
Платная        23
Name: count, dtype: int64
Сохранено: parking_almaty.xlsx  (600 строк)
CSV также сохранён.
